# Ingestion Pipeline

End-to-end pipeline from raw PubMed abstracts to populated vector databases.

**Step 1 — Load & Chunk:** Load raw `.txt` files via LangChain DirectoryLoader, filter out multilingual abstracts, strip headers/footers, and chunk with RecursiveCharacterTextSplitter.  
**Step 2 — Ingest:** Push chunks into multiple vector databases (ChromaDB, Qdrant, LanceDB, Weaviate).

**Input:** `data/raw/{pubid}.txt`  
**Output:** Populated vector stores in `vectorstores/`

In [3]:
import sys
sys.path.append("..")

import os
import re
from langdetect import detect, LangDetectException
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
import config
import uuid
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "true"
from google.colab import userdata

## Parsing & Cleaning

Two minimal operations:
- **`detect_multilingual`** — flags abstracts with a non-English `Publisher:` translation block (common in bilingual journals like Canadian ones). These are excluded from the KB.
- **`extract_abstract_body`** — strips the PubMed header (citation line, title, authors, affiliations) and footer (DOI, PMID, translated blocks), leaving only the abstract text.

In [4]:
FOOTER_PATTERN = re.compile(
    r"\n(DOI:|PMID:|PMCID:|Published by|Publisher:|Copyright).*",
    re.DOTALL,
)


def detect_multilingual(raw_text):
    """Return True if the abstract has a non-English Publisher: translation block."""
    publisher_match = re.search(r"^Publisher:", raw_text, re.MULTILINE)
    if not publisher_match:
        return False
    excerpt = raw_text[publisher_match.end() : publisher_match.end() + 400].strip()
    if not excerpt:
        return False
    try:
        return detect(excerpt) != "en"
    except LangDetectException:
        return False


def extract_abstract_body(raw_text):
    """Strip PubMed header and footer; return only the abstract body text."""
    # Cut at the first footer marker
    text = FOOTER_PATTERN.sub("", raw_text).strip()

    paragraphs = [p.strip() for p in re.split(r"\n{2,}", text) if p.strip()]

    body_start = len(paragraphs)
    for i, para in enumerate(paragraphs):
        # Structured abstract: BACKGROUND:, METHODS:, RESULTS:, etc.
        if re.match(r"[A-Z][A-Z ]+:", para):
            body_start = i
            break
        # Unstructured abstract with explicit heading
        if para.strip().lower() == "abstract":
            body_start = i + 1
            break
        # After the 3-para header (citation / title / authors),
        # the first long paragraph that isn't an affiliations block
        if i >= 3 and len(para) > 150 and not re.search(r"\(\d+\)", para[:80]):
            body_start = i
            break

    return "\n".join(paragraphs[body_start:])

## Chunking & Loading

`load_and_chunk_abstracts` does everything in one pass: load via LangChain DirectoryLoader → filter multilingual → extract body → chunk. Returns `(documents, ids, multilingual_pubids)`.

In [ ]:
def get_text_splitter(chunk_size=None, chunk_overlap=None, separators=None):
    """Return a RecursiveCharacterTextSplitter configured from config defaults or overrides."""
    return RecursiveCharacterTextSplitter(
        chunk_size=chunk_size or config.CHUNK_SIZE,
        chunk_overlap=chunk_overlap or config.CHUNK_OVERLAP,
        separators=separators or config.CHUNK_SEPARATORS,
    )


def load_and_chunk_abstracts(abstracts_dir, text_splitter=None):
    """Load all .txt abstracts, filter multilingual, extract body text, and chunk.

    Args:
        abstracts_dir: Directory containing one .txt file per PubMed abstract.
        text_splitter: Optional pre-configured splitter; defaults to get_text_splitter().

    Returns:
        documents: List of LangChain Document objects ready for ingestion.
        ids: Parallel list of document IDs (f"{pubid}_chunk{n}").
        multilingual_pubids: PubIDs excluded because they contained a non-English
                             Publisher: translation block.
    """
    if text_splitter is None:
        text_splitter = get_text_splitter()

    loader = DirectoryLoader(
        str(abstracts_dir),
        glob="*.txt",
        loader_cls=TextLoader,
        loader_kwargs={"encoding": "utf-8"},
        show_progress=False,
        silent_errors=True,
    )
    raw_docs = loader.load()
    print(f"Loaded {len(raw_docs)} abstract files")

    documents = []
    ids = []
    multilingual_pubids = []
    skipped = []

    for doc in raw_docs:
        raw_text = doc.page_content
        pubid = os.path.basename(doc.metadata["source"]).replace(".txt", "")

        if detect_multilingual(raw_text):
            multilingual_pubids.append(pubid)
            continue

        body = extract_abstract_body(raw_text)
        if not body or len(body) < 100:
            skipped.append(pubid)
            continue

        chunks = text_splitter.split_text(body)
        for chunk_idx, chunk in enumerate(chunks):
            doc_id = f"{pubid}_chunk{chunk_idx}"
            chunk_id = str(uuid.uuid5(uuid.NAMESPACE_DNS, doc_id))
            documents.append(Document(
                page_content=chunk,
                metadata={
                    "pubid": pubid,
                    "source": doc.metadata["source"],
                    "chunk_index": chunk_idx,
                    "total_chunks": len(chunks),
                },
                id=chunk_id,
            ))
            ids.append(chunk_id)

    n_ingested = len(raw_docs) - len(multilingual_pubids) - len(skipped)
    print(f"Created {len(documents)} chunks from {n_ingested} abstracts")
    if multilingual_pubids:
        print(f"Excluded {len(multilingual_pubids)} multilingual abstracts")
    if skipped:
        print(f"Skipped {len(skipped)} abstracts with no extractable body")

    return documents, ids, multilingual_pubids

## Ingestion Functions

One function per vector database. Each is idempotent — skips ingestion if the store already exists.

In [ ]:
def ingest_to_chroma(documents, ids, embeddings, db_name="pubmed_chroma"):
    """Ingest documents into a local ChromaDB collection, or load it if it already exists.

    Args:
        documents: LangChain Document objects to index.
        ids: Document IDs parallel to documents.
        embeddings: LangChain embeddings instance used to encode chunks.
        db_name: ChromaDB collection name and default subdirectory name.
        persist_dir: Override the storage path (defaults to vectorstores/{db_name}).

    Returns:
        Chroma vector store instance.
    """
    from langchain_chroma import Chroma
    persist_dir = str(config.VECTORSTORE_DIR / db_name)
    print(persist_dir)

    vector_store = Chroma(
        collection_name=db_name,
        persist_directory=persist_dir,
        embedding_function=embeddings,
    )

    if vector_store._collection.count() == 0:
        vector_store.add_documents(documents=documents, ids=ids)
        print(f"Total: {len(documents)} chunks ingested into ChromaDB")
    else:
        print(f"Loaded existing ChromaDB from {persist_dir}")

    return vector_store


def ingest_to_qdrant(documents, ids, embeddings, collection_name="pubmed_qdrant", path=None):
    """Ingest documents into a local Qdrant collection, or load it if it already exists.

    Args:
        documents: LangChain Document objects to index.
        ids: Document IDs parallel to documents.
        embeddings: LangChain embeddings instance used to encode chunks.
        collection_name: Qdrant collection name and default subdirectory name.
        path: Override the storage path (defaults to vectorstores/{collection_name}).

    Returns:
        QdrantVectorStore instance.
    """
    from langchain_qdrant import QdrantVectorStore
    from qdrant_client import QdrantClient
    from qdrant_client.models import Distance, VectorParams

    path = path or str(config.VECTORSTORE_DIR / collection_name)

    sample_embedding = embeddings.embed_query("test")
    embedding_dim = len(sample_embedding)

    client = QdrantClient(path=path)

    if not client.collection_exists(collection_name):
        client.create_collection(
            collection_name=collection_name,
            vectors_config=VectorParams(size=embedding_dim, distance=Distance.COSINE),
        )

    vector_store = QdrantVectorStore(
        client=client,
        collection_name=collection_name,
        embedding=embeddings,
    )

    existing = client.get_collection(collection_name).points_count
    if existing == 0:
        vector_store.add_documents(documents=documents, ids=ids)
        print(f"Ingested {len(documents)} chunks into Qdrant")
    else:
        print(f"Loaded existing Qdrant collection ({existing} points)")

    return vector_store


def ingest_to_lancedb(documents, ids, embeddings, table_name="pubmed_lance", path=None):
    """Ingest documents into a local LanceDB table, or load it if it already exists.

    LanceDB stores vectors alongside raw text and metadata in a columnar format,
    so embeddings are computed explicitly before writing.

    Args:
        documents: LangChain Document objects to index.
        ids: Document IDs parallel to documents.
        embeddings: LangChain embeddings instance used to encode chunks.
        table_name: LanceDB table name and default subdirectory key.
        path: Override the storage path (defaults to vectorstores/lancedb).

    Returns:
        Tuple of (LanceDB connection, table).
    """
    import lancedb

    path = path or str(config.VECTORSTORE_DIR / "lancedb")
    db = lancedb.connect(path)

    texts = [doc.page_content for doc in documents]
    metadatas = [doc.metadata for doc in documents]

    if table_name not in db.table_names():
        vectors = embeddings.embed_documents(texts)
        data = [
            {"id": doc_id, "text": text, "vector": vec, **meta}
            for doc_id, text, vec, meta in zip(ids, texts, vectors, metadatas)
        ]
        table = db.create_table(table_name, data=data)
        print(f"Ingested {len(documents)} chunks into LanceDB")
    else:
        table = db.open_table(table_name)
        print(f"Loaded existing LanceDB table '{table_name}' ({table.count_rows()} rows)")

    return db, table


def ingest_to_faiss(documents, embeddings, index_name="pubmed_faiss", path=None):
    """Ingest documents into a local FAISS index, or load it if it already exists.

    FAISS uses its own internal integer indices so no ids parameter is needed.
    The index is saved as index.faiss + index.pkl under vectorstores/{index_name}/.

    Args:
        documents: LangChain Document objects to index.
        embeddings: LangChain embeddings instance used to encode chunks.
        index_name: Subdirectory name under vectorstores/ for the saved index.
        path: Override the storage path.

    Returns:
        FAISS vector store instance.
    """
    from langchain_community.vectorstores import FAISS

    save_path = path or str(config.VECTORSTORE_DIR / index_name)

    if os.path.exists(save_path) and os.listdir(save_path):
        vector_store = FAISS.load_local(
            save_path, embeddings, allow_dangerous_deserialization=True
        )
        print(f"Loaded existing FAISS index from {save_path} ({len(vector_store.index_to_docstore_id)} vectors)")
    else:
        os.makedirs(save_path, exist_ok=True)
        vector_store = FAISS.from_documents(documents, embeddings)
        vector_store.save_local(save_path)
        print(f"Ingested {len(documents)} chunks into FAISS, saved to {save_path}")

    return vector_store


def ingest_to_weaviate(documents, ids, embeddings, index_name='pubmed_weaviate', api_key=None):
    """Ingest documents into a Weaviate Cloud collection.

    Requires WEAVIATE_URL and WEAVIATE_API_KEY set in the environment (or .env).
    Unlike the local stores this function always re-ingests; manage idempotency
    via Weaviate's own deduplication or by checking the collection count first.

    Args:
        documents: LangChain Document objects to index.
        ids: Document IDs parallel to documents.
        embeddings: LangChain embeddings instance used to encode chunks.
        url: Weaviate cluster URL (defaults to WEAVIATE_URL env var).
        api_key: Weaviate API key (defaults to WEAVIATE_API_KEY env var).

    Returns:
        Tuple of (WeaviateVectorStore, weaviate client).
    """
    import weaviate
    from weaviate.classes.init import Auth
    from langchain_weaviate import WeaviateVectorStore

    url = "https://fiuznxyotgul4dwnhuymg.c0.asia-southeast1.gcp.weaviate.cloud"
    api_key = api_key or os.getenv("WEAVIATE_KEY", "")
    api_key = api_key or userdata.get("WEAVIATE_KEY")

    client = weaviate.connect_to_weaviate_cloud(
        cluster_url=url,
        auth_credentials=Auth.api_key(api_key),
    )

    vector_store = WeaviateVectorStore(
        client=client,
        index_name=index_name,
        text_key="text",
        embedding=embeddings,
    )

    vector_store.add_documents(documents=documents, ids=ids)
    print(f"Ingested {len(documents)} chunks into Weaviate")

    return vector_store, client

---
## Step 1: Load, Clean & Chunk

In [7]:
text_splitter = get_text_splitter()
documents, ids, multilingual_pubids = load_and_chunk_abstracts(
    config.DATA_RAW_DIR, text_splitter
)

Loaded 1000 abstract files
Created 2849 chunks from 999 abstracts
Excluded 1 multilingual abstracts


### Multilingual Abstracts Excluded

These pubids had a non-English `Publisher:` block and were skipped. Keep this list if you want to exclude them from evaluation too.

In [8]:
print(f"Multilingual abstracts excluded from KB ({len(multilingual_pubids)} total):")
print(multilingual_pubids)

Multilingual abstracts excluded from KB (1 total):
['24666444']


### Sample Chunks

In [9]:
for doc in documents[:3]:
    print(f"ID:      {doc.id}")
    print(f"Chunk:   {doc.metadata['chunk_index'] + 1} of {doc.metadata['total_chunks']}")
    print(f"Content: {doc.page_content[:300]}")
    print()

ID:      6e3d4697-068e-5bac-a69b-352a7da5ded8
Chunk:   1 of 4
Content: BACKGROUND: Programmed cell death (PCD) is the regulated death of cells within 
an organism. The lace plant (Aponogeton madagascariensis) produces perforations 
in its leaves through PCD. The leaves of the plant consist of a latticework of 
longitudinal and transverse veins enclosing areoles. PCD oc

ID:      708d2223-6950-59b6-8b25-856ecbc2ea62
Chunk:   2 of 4
Content: areole within a window stage leaf (PCD is occurring) was divided into three 
areas based on the progression of PCD; cells that will not undergo PCD (NPCD), 
cells in early stages of PCD (EPCD), and cells in late stages of PCD (LPCD). 
Window stage leaves were stained with the mitochondrial dye MitoT

ID:      7872f56f-b55f-5961-981a-b4607dbf5256
Chunk:   3 of 4
Content: transition pore (PTP) formation during PCD was indirectly examined via in vivo 
cyclosporine A (CsA) treatment. This treatment resulted in lace plant leaves 
with a significantly lowe

---
## Step 2: Ingest into Vector Databases

Same chunk corpus pushed into multiple databases for fair cross-DB comparison.

In [10]:
embedding_key = config.DEFAULT_EMBEDDING
embeddings = HuggingFaceEmbeddings(model_name=config.EMBEDDING_MODELS[embedding_key])
print(f"Using embedding model: {config.EMBEDDING_MODELS[embedding_key]}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Using embedding model: sentence-transformers/all-MiniLM-L6-v2


### ChromaDB

In [11]:
chroma_store = ingest_to_chroma(
    documents=documents, ids=ids, embeddings=embeddings,
    db_name=f"{embedding_key}_pubmed_chroma"
)

/content/vectorstores/minilm_pubmed_chroma
Total: 2849 chunks ingested into ChromaDB


### Qdrant

In [12]:
qdrant_store = ingest_to_qdrant(
    documents, ids, embeddings,
    collection_name=f"{embedding_key}_pubmed_qdrant"
)

Ingested 2849 chunks into Qdrant


### LanceDB

In [13]:
lance_db, lance_table = ingest_to_lancedb(
    documents, ids, embeddings,
    table_name=f"{embedding_key}_pubmed_lance"
)

/tmp/ipykernel_108855/668128584.py:64: DeprecationWarning: table_names() is deprecated, use list_tables() instead
  if table_name not in db.table_names():


Ingested 2849 chunks into LanceDB


## FAISS

In [14]:
faiss_store = ingest_to_faiss(
    documents, embeddings,
    index_name=f"{embedding_key}_pubmed_faiss"
)

Ingested 2849 chunks into FAISS, saved to /content/vectorstores/minilm_pubmed_faiss


### Weaviate Cloud

Requires `WEAVIATE_KEY` in `.env`.

In [15]:
weaviate_store, weaviate_client = ingest_to_weaviate(documents, ids, embeddings, f"{embedding_key}_pubmed_weaviate")

Ingested 2849 chunks into Weaviate


### Verify Ingestion Counts

In [16]:
collection = weaviate_client.collections.get(f"{embedding_key}_pubmed_weaviate")
print(f"ChromaDB: {chroma_store._collection.count()} chunks")
print(f"FAISS:    {len(faiss_store.index_to_docstore_id)} chunks")
print(f"LanceDB:  {lance_table.count_rows()} chunks")
print(f"Weaviate: {collection.aggregate.over_all().total_count} chunks")
print(f"\nAll stores ingested from {len(documents)} source documents")

ChromaDB: 2849 chunks
FAISS:    2849 chunks
LanceDB:  2849 chunks
Weaviate: 2849 chunks

All stores ingested from 2849 source documents
